# Exact K-scaling experiment

cell 1 - imports

In [1]:
import os, json, time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cpu


In [2]:
def simulate_global_state_ou(
        T=20000, D=256, seed=123, 
        mu=-1.5, sigma=0.3, 
        sigma_drift=0.001,
        lambda_ou = 0.03,        # mean-reversion strength (0 -> random walk) - A good starting value is 0.02 to 0.05.
        eps0 = 0.0               # long-run mean of eps
):
    """
    Ou drift:
    eps_t = (1 - lambda_ou) * eps_{t-1} + lambda_ou * eps0 + Normal(0, sigma_drift)
    then eps enters the logistic transition probabilities.
    """
    rng = np.random.default_rng(seed)

    alpha = rng.normal(mu, sigma, size=D)
    beta  = rng.normal(mu, sigma, size=D)

    eps = np.zeros(D) + eps0
    X = np.zeros((T, D), dtype=np.int8)
    X[0] = rng.binomial(1, 0.5, size=D)

    for t in range(1, T):
        # OU update (mean-reverting)
        eps = (1.0 - lambda_ou) * eps + lambda_ou * eps0 + rng.normal(0.0, sigma_drift, size=D)

        # stable logistic
        logit_p = np.clip(alpha + eps, -20, 20)
        logit_q = np.clip(beta  + eps, -20, 20)
        p = 1 / (1 + np.exp(-logit_p))
        q = 1 / (1 + np.exp(-logit_q))

        X[t] = rng.binomial(1, np.where(X[t-1]==0, p, 1-q))

    return X


In [3]:
def construct_nodes(D=256, K=40, subset_size=25, overlap_target=5, seed=123):
    rng = np.random.default_rng(seed)
    subsets = [set(rng.choice(D, subset_size, replace=False))]
    for _ in range(1, K):
        base_subset = subsets[rng.integers(len(subsets))]
        base = rng.choice(list(base_subset), overlap_target, replace=False)
        rest = rng.choice(list(set(range(D)) - set(base)), subset_size - overlap_target, replace=False)
        subsets.append(set(base) | set(rest))
    return subsets

def compute_node_signals(X, subsets):
    Z = np.zeros((X.shape[0], len(subsets)), dtype=np.float32)
    for k, S in enumerate(subsets):
        Z[:, k] = X[:, list(S)].mean(axis=1)
    return Z


cell 4 - helper funcs to train for a given K on clean Z (OU process with no dropout)

In [4]:
class MaskedWindowDataset(Dataset):
    def __init__(self, Z, window=128, mask_ratio=0.15, seed=2):
        rng = np.random.default_rng(seed)
        T, K = Z.shape
        self.Z = Z.astype(np.float32)
        self.window = window
        self.N = T - window
        self.masks = rng.random((self.N, window, K)) < mask_ratio

    def __len__(self):
        return self.N

    def __getitem__(self, idx):
        x = self.Z[idx:idx+self.window].copy()
        m = self.masks[idx]
        y = x.copy()
        x[m] = 0.0
        return torch.from_numpy(x), torch.from_numpy(m), torch.from_numpy(y)

class PairedMaskedWindowDataset(Dataset):
    """
    Inputs come from corrupted Z_in, but targets are clean Z_true.
    Mask m defines which positions are evaluated (and also zeroed in x).
    """
    def __init__(self, Z_in, Z_true, window=128, mask_ratio=0.15, seed=2):
        assert Z_in.shape == Z_true.shape
        rng = np.random.default_rng(seed)

        T, K = Z_true.shape
        self.Z_in = Z_in.astype(np.float32)
        self.Z_true = Z_true.astype(np.float32)
        self.window = window
        self.N = T - window
        self.masks = rng.random((self.N, window, K)) < mask_ratio

    def __len__(self):
        return self.N
    
    def __getitem__(self, idx):
        x = self.Z_in[idx:idx+self.window].copy()   # corrupted context
        y = self.Z_true[idx:idx+self.window].copy() # clean target
        m = self.masks[idx]

        x[m] = 0.0  # standard masked modeling
        return torch.from_numpy(x), torch.from_numpy(m), torch.from_numpy(y)



In [5]:
def destroy_dropout_Z(Z, delta, seed=999, mode="node", block_len=16):
    """
    Dropout/Corruption applied to Z (node signals).
    Returns Z_in (Corrupted) from Z (clean).
    mode:
    - "element": drop ranom entries independly.
    - "node": drop whole nodes (columns) across all time.
    - "block": drop contiguous time blocks for all nodes.     
    """
    rng = np.random.default_rng(seed)
    Z = Z.astype(np.float32)
    Z_in = Z.copy()
    T, K = Z.shape

    if mode == "element":
        drop = rng.random((T, K)) < delta
        Z_in[drop] = 0.0

    elif mode == "node":
        drop_cols = rng.random(K) < delta
        Z_in[:, drop_cols] = 0.0

    elif mode == "block":
        t = 0
        while t < T:
            if rng.random() < delta:
                Z_in[t:min(T, t + block_len), :] = 0.0
            t += block_len
    
    else:
        raise ValueError("mode must be one of: element, node, block")
    
    return Z_in

In [6]:
class MaskedTimeTransformer(nn.Module):
    def __init__(self, K, window=128, d_model=128, nhead=4, num_layers=3, dim_ff=256, dropout=0.1):
        super().__init__()
        self.in_proj = nn.Linear(K, d_model)
        self.pos_emb = nn.Parameter(torch.randn(window, d_model) * 0.02)

        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True,
            activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.out_proj = nn.Linear(d_model, K)

    def forward(self, x):
        B, L, _ = x.shape
        h = self.in_proj(x) + self.pos_emb[:L].unsqueeze(0)
        h = self.encoder(h)
        return self.out_proj(h)


In [7]:
@torch.no_grad()
def collect_masked_predictions(model, loader, device):
    model.eval()
    z_true, z_pred = [], []
    for xb, mb, yb in loader:
        xb = xb.to(device).float()
        m = mb.numpy().astype(bool)
        y = yb.numpy().astype(np.float32)
        yhat = model(xb).cpu().numpy().astype(np.float32)
        z_true.append(y[m])
        z_pred.append(yhat[m])
    return np.concatenate(z_true), np.concatenate(z_pred)

def baseline_predict_window_node_mean(x_masked_np):
    node_means = x_masked_np.mean(axis=1, keepdims=True)  # (B,1,K)
    return np.repeat(node_means, x_masked_np.shape[1], axis=1)


def collect_baseline_window_predictions(val_loader):
    z_true, z_pred = [], []
    for xb, mb, yb in val_loader:
        x = xb.numpy().astype(np.float32)
        m = mb.numpy().astype(bool)
        y = yb.numpy().astype(np.float32)
        yhat = baseline_predict_window_node_mean(x)
        z_true.append(y[m])
        z_pred.append(yhat[m])
    return np.concatenate(z_true), np.concatenate(z_pred)

def discretize(x, bins):
    return np.digitize(x, bins) - 1

def entropy_from_counts(counts):
    p = counts / counts.sum()
    p = p[p > 0]
    return float(-np.sum(p * np.log2(p)))

def mutual_information(x, y):
    xy = np.stack([x, y], axis=1)
    _, cxy = np.unique(xy, axis=0, return_counts=True)
    _, cx = np.unique(x, return_counts=True)
    _, cy = np.unique(y, return_counts=True)
    Hx = entropy_from_counts(cx)
    Hy = entropy_from_counts(cy)
    Hxy = entropy_from_counts(cxy)
    return Hx + Hy - Hxy

def H_from_samples(x):
    _, cx = np.unique(x, return_counts=True)
    return entropy_from_counts(cx)

In [8]:
CFG = dict(
    # data process (fixed across all K)
    T=20000,
    D=256,
    lambda_ou=0.03,
    sigma_base=0.001,
    seed_world=123,

    # representation
    subset_seed=123,     # seed for construct_nodes
    subset_size=25,      # only used if your construct_nodes supports it
    overlap_target=5,    # only used if your construct_nodes supports it

    # model + dataset
    window=128,
    mask_ratio=0.15,
    split_frac=0.8,
    batch_size=64,
    epochs=5,
    lr=3e-4,

    # transformer hyperparams (must be consistent across K)
    d_model=128,
    nhead=4,
    num_layers=3,
    dim_ff=256,
    dropout=0.1,

    # evaluation (information theory)
    bins=np.linspace(0, 1, 21),

    # destruction (this is your “phase-transition-like” test)
    dropout_mode="block",
    block_len=64,
    deltas=np.linspace(0.0, 0.5, 21),  # denser up to 0.5 like you did

    # scaling axis
    K_list=[20, 40, 80, 160],          # reduce if too slow

    # output
    out_dir="k_scaling_results",
)

os.makedirs(CFG["out_dir"], exist_ok=True)
CFG  # display


{'T': 20000,
 'D': 256,
 'lambda_ou': 0.03,
 'sigma_base': 0.001,
 'seed_world': 123,
 'subset_seed': 123,
 'subset_size': 25,
 'overlap_target': 5,
 'window': 128,
 'mask_ratio': 0.15,
 'split_frac': 0.8,
 'batch_size': 64,
 'epochs': 5,
 'lr': 0.0003,
 'd_model': 128,
 'nhead': 4,
 'num_layers': 3,
 'dim_ff': 256,
 'dropout': 0.1,
 'bins': array([0.  , 0.05, 0.1 , 0.15, 0.2 , 0.25, 0.3 , 0.35, 0.4 , 0.45, 0.5 ,
        0.55, 0.6 , 0.65, 0.7 , 0.75, 0.8 , 0.85, 0.9 , 0.95, 1.  ]),
 'dropout_mode': 'block',
 'block_len': 64,
 'deltas': array([0.   , 0.025, 0.05 , 0.075, 0.1  , 0.125, 0.15 , 0.175, 0.2  ,
        0.225, 0.25 , 0.275, 0.3  , 0.325, 0.35 , 0.375, 0.4  , 0.425,
        0.45 , 0.475, 0.5  ]),
 'K_list': [20, 40, 80, 160],
 'out_dir': 'k_scaling_results'}

In [9]:
X_clean = simulate_global_state_ou(
    T=CFG["T"],
    D=CFG["D"],
    seed=CFG["seed_world"],
    sigma_drift=CFG["sigma_base"],
    lambda_ou=CFG["lambda_ou"],
)

print("X_clean shape:", X_clean.shape, "mean:", X_clean.mean())


X_clean shape: (20000, 256) mean: 0.5089306640625


In [10]:
def train_model_for_K(Z_clean, K, cfg, device):
    """
    Train transformer on clean Z (no dropout-on-Z here).
    Masking is still used as the training objective (masked modeling).
    """
    window = cfg["window"]
    mask_ratio = cfg["mask_ratio"]
    batch_size = cfg["batch_size"]
    epochs = cfg["epochs"]
    lr = cfg["lr"]

    # split (no leakage)
    Tz = Z_clean.shape[0]
    split = int(cfg["split_frac"] * Tz)
    Z_train = Z_clean[:split]
    Z_val   = Z_clean[split - window:]

    train_ds = MaskedWindowDataset(Z_train, window=window, mask_ratio=mask_ratio, seed=1)
    val_ds   = MaskedWindowDataset(Z_val,   window=window, mask_ratio=mask_ratio, seed=2)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False)

    # model (K changes input/output dims!)
    model = MaskedTimeTransformer(
        K=K,
        window=window,
        d_model=cfg["d_model"],
        nhead=cfg["nhead"],
        num_layers=cfg["num_layers"],
        dim_ff=cfg["dim_ff"],
        dropout=cfg["dropout"],
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss(reduction="none")

    history = {"train_masked_mse": [], "val_masked_mse": []}

    for ep in range(1, epochs + 1):
        model.train()
        tr = 0.0
        for xb, mb, yb in train_loader:
            xb = xb.to(device).float()
            yb = yb.to(device).float()
            mb = mb.to(device)

            optimizer.zero_grad()
            yhat = model(xb)
            loss = loss_fn(yhat, yb)
            loss = (loss * mb).sum() / (mb.sum() + 1e-8)
            loss.backward()
            optimizer.step()
            tr += loss.item()
        tr /= len(train_loader)

        model.eval()
        va = 0.0
        with torch.no_grad():
            for xb, mb, yb in val_loader:
                xb = xb.to(device).float()
                yb = yb.to(device).float()
                mb = mb.to(device)
                yhat = model(xb)
                loss = loss_fn(yhat, yb)
                loss = (loss * mb).sum() / (mb.sum() + 1e-8)
                va += loss.item()
        va /= len(val_loader)

        history["train_masked_mse"].append(tr)
        history["val_masked_mse"].append(va)
        print(f"K={K:3d} | epoch {ep:02d} | train {tr:.6f} | val {va:.6f}")

    return model, history, split


In [11]:
def eval_dropout_sweep_for_K(model, Z_clean, split, K, cfg, device):
    window = cfg["window"]
    mask_ratio = cfg["mask_ratio"]
    bins = cfg["bins"]
    mode = cfg["dropout_mode"]
    block_len = cfg["block_len"]
    deltas = cfg["deltas"]
    batch_size = cfg["batch_size"]

    results = []

    for delta in deltas:
        # corrupt input Z -> Z_in (targets remain clean!)
        Z_in = destroy_dropout_Z(Z_clean, delta, seed=999, mode=mode, block_len=block_len)

        Z_true_val = Z_clean[split - window:]
        Z_in_val   = Z_in[split - window:]

        val_ds = PairedMaskedWindowDataset(Z_in_val, Z_true_val, window=window, mask_ratio=mask_ratio, seed=2)
        val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

        # transformer
        zt_tf, zp_tf = collect_masked_predictions(model, val_loader, device)
        zt_tf_d = discretize(zt_tf, bins)
        zp_tf_d = discretize(zp_tf, bins)

        H_true = H_from_samples(zt_tf_d)
        MI_tf = mutual_information(zt_tf_d, zp_tf_d)
        R_tf = MI_tf / H_true if H_true > 1e-12 else 0.0

        # baseline
        zt_b, zp_b = collect_baseline_window_predictions(val_loader)
        zt_b_d = discretize(zt_b, bins)
        zp_b_d = discretize(zp_b, bins)

        MI_b = mutual_information(zt_b_d, zp_b_d)
        R_b = MI_b / H_true if H_true > 1e-12 else 0.0

        results.append({
            "K": int(K),
            "delta": float(delta),
            "mode": mode,
            "block_len": int(block_len) if mode == "block" else None,
            "H_true": float(H_true),
            "MI_transformer": float(MI_tf),
            "R_transformer": float(R_tf),
            "MI_baseline_window": float(MI_b),
            "R_baseline_window": float(R_b),
        })

    return results


In [ ]:
results_by_K = {}
models_saved = {}

t0 = time.time()

for K in CFG["K_list"]:
    print("\n" + "="*80)
    print("K =", K)

    # 1) build subsets and Z_clean for this K
    # NOTE: if your construct_nodes supports subset_size/overlap_target, add them here.
    #subsets = construct_nodes(D=CFG["D"], K=K, seed=CFG["subset_seed"])

    subsets = construct_nodes(
    D=CFG["D"],
    K=K,
    subset_size=CFG["subset_size"],
    overlap_target=CFG["overlap_target"],
    seed=CFG["subset_seed"],)

    Z_clean = compute_node_signals(X_clean, subsets)

    # 2) train model for this K on clean OU Z
    model, history, split = train_model_for_K(Z_clean, K, CFG, device)

    # 3) save model + training history
    model_path = os.path.join(
        CFG["out_dir"],
        f"model_ou_lam{CFG['lambda_ou']}_sig{CFG['sigma_base']}_win{CFG['window']}_K{K}.pt"
    )
    torch.save(model.state_dict(), model_path)

    hist_path = os.path.join(
        CFG["out_dir"],
        f"history_ou_lam{CFG['lambda_ou']}_sig{CFG['sigma_base']}_win{CFG['window']}_K{K}.json"
    )
    with open(hist_path, "w", encoding="utf-8") as f:
        json.dump(history, f, indent=2)

    models_saved[K] = model_path
    print("saved model:", model_path)
    print("saved history:", hist_path)

    # 4) evaluate dropout sweep for this K
    resK = eval_dropout_sweep_for_K(model, Z_clean, split, K, CFG, device)
    results_by_K[K] = resK

    # 5) save results for this K
    res_path = os.path.join(
        CFG["out_dir"],
        f"dropoutZ_blocklen{CFG['block_len']}_win{CFG['window']}_K{K}.json"
    )
    payload = {"CFG": {**CFG, "bins": None}, "model_path": model_path, "results": resK}
    # bins are numpy; store separately if you want
    
    payload["CFG"]["bins_edges"] = CFG["bins"].tolist()
    payload["CFG"]["deltas"] = np.array(CFG["deltas"]).tolist()   # <-- add this line
    
    payload["CFG"].pop("bins", None)
        
    with open(res_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)

    print("saved sweep:", res_path)

print("\nDONE. Total minutes:", (time.time() - t0) / 60.0)
models_saved



K = 20
K= 20 | epoch 01 | train 0.025438 | val 0.010087
K= 20 | epoch 02 | train 0.012868 | val 0.009563
K= 20 | epoch 03 | train 0.010705 | val 0.009403
K= 20 | epoch 04 | train 0.009932 | val 0.009303
K= 20 | epoch 05 | train 0.009632 | val 0.009242
saved model: k_scaling_results\model_ou_lam0.03_sig0.001_win128_K20.pt
saved history: k_scaling_results\history_ou_lam0.03_sig0.001_win128_K20.json


TypeError: Object of type ndarray is not JSON serializable

In [ ]:
plt.figure()
for K in CFG["K_list"]:
    res = results_by_K[K]
    deltas = [r["delta"] for r in res]
    R = [r["R_transformer"] for r in res]
    plt.plot(deltas, R, marker="o", label=f"K={K}")

plt.xlabel("δ (block dropout probability)")
plt.ylabel("Retention R = I/H")
plt.title(f"Retention vs δ (block_len={CFG['block_len']}, window={CFG['window']}, OU λ={CFG['lambda_ou']})")
plt.legend()
plt.savefig(f"Retention vs δ for K={K}.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
def extract_thresholds(results):
    # assumes results sorted by delta
    deltas = np.array([r["delta"] for r in results], dtype=float)
    R = np.array([r["R_transformer"] for r in results], dtype=float)
    R0 = R[0]

    def first_delta_below(frac):
        target = frac * R0
        idx = np.where(R <= target)[0]
        return float(deltas[idx[0]]) if len(idx) else None

    d50 = first_delta_below(0.5)
    d90 = first_delta_below(0.9)
    d10 = first_delta_below(0.1)
    width = (d10 - d90) if (d10 is not None and d90 is not None) else None

    return dict(R0=float(R0), delta50=d50, delta90=d90, delta10=d10, width=width)

summary = []
for K in CFG["K_list"]:
    th = extract_thresholds(results_by_K[K])
    summary.append({"K": K, **th})

summary


In [ ]:
Ks = np.array([s["K"] for s in summary], dtype=float)
widths = np.array([s["width"] if s["width"] is not None else np.nan for s in summary], dtype=float)

plt.figure()
plt.plot(Ks, widths, marker="o")
plt.xscale("log")
plt.xlabel("K (log scale)")
plt.ylabel("width = δ10 − δ90")
plt.title("Knee width vs K (transition sharpening test)")
plt.savefig(f"Knee width vs K (transition sharpening test).png", dpi=200, bbox_inches="tight")
plt.show()

summary
